# RAG Pipeline Notebook

Tests PDF extraction, chunking, embedding, retrieval, and refusal behavior step by step.

In [ ]:
import pymupdf
from pathlib import Path

pdf = Path('corpus/NIST_AI_RMF_1.0.pdf')
with pymupdf.open(pdf) as document:
    text = '\n'.join(page.get_text('text') for page in document)
print(pdf.name, len(document), len(text))

In [ ]:
from app.ingestion import clean_text, split_text
pages = [(number, clean_text(page.get_text('text'))) for number, page in enumerate(pymupdf.open(pdf), start=1)]
chunks = split_text(pages, 800, 150)
print(len(chunks), chunks[0]['page'], chunks[0]['text'][:200])

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('intfloat/e5-small-v2')
query = model.encode(['query: What are the AI RMF functions?'], normalize_embeddings=True)
passages = model.encode(['passage: ' + chunk['text'] for chunk in chunks[:20]], normalize_embeddings=True)
scores = passages @ query[0]
print(sorted([(float(score), index) for index, score in enumerate(scores)], reverse=True)[:5])